# CS383: Data Science and Machine Learning
## Lecture 5 — Exploratory Data Analysis and Foundational Statistics

**Guiding question:** before you trust a chart or a model, can you describe what's actually in your data — its center, its spread, its shape, and how (if at all) its variables relate to each other?

Today uses two real NYC datasets: **restaurant inspections** (numeric-rich — a good fit for statistics and correlation mechanics) and **NYC 311 complaints** (the dataset you already know, for applied EDA and communicating a finding to a non-technical reader — exactly what Assignment 2 asks you to do).

---
**Live in-class version.** Type along at each `__________` blank — everything else is filled in so class time stays on the new syntax, not on retyping boilerplate.

---

## Part 1 — Descriptive Statistics

Before any chart, a few numbers already tell you a lot about a dataset: where its center is, how spread out it is, and where the middle 50% of it lives.

### Setup — NYC restaurant inspections

We pull inspection-level violation records and aggregate them to one row per inspection: how many violations were cited, how many were `"Critical"`, and the resulting numeric `score` (a higher score means *more* violations).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

SOCRATA_URL = "https://data.cityofnewyork.us/resource/43nn-pn8j.json"

try:
    response = requests.get(
        SOCRATA_URL,
        params={
            "$limit": 15000,
            "$order": "inspection_date DESC",
            "$select": "camis,boro,cuisine_description,inspection_date,score,grade,"
                        "violation_code,critical_flag",
        },
        timeout=10,
    )
    response.raise_for_status()
    raw = pd.DataFrame(response.json())
    raw["score"] = pd.to_numeric(raw["score"], errors="coerce")

    inspections_df = (
        raw.groupby(["boro", "cuisine_description", "score", "grade"], dropna=False)
        .agg(
            violation_count=("violation_code", "count"),
            critical_count=("critical_flag", lambda s: (s == "Critical").sum()),
        )
        .reset_index()
    )
    inspections_df = inspections_df.dropna(subset=["score"]).reset_index(drop=True)
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 1200
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza",
                       "Japanese", "Caribbean", "Bakery", "Coffee/Tea", "Chicken"]

    # Score is built FROM violations here, on purpose -- see Part 4.
    violation_count = rng.poisson(lam=3, size=n)
    critical_count = rng.binomial(violation_count, 0.4)
    noise = rng.normal(0, 3, size=n)
    score = np.clip(violation_count * 7 + critical_count * 5 + noise, 0, 140).round().astype(int)

    grade = np.where(score <= 13, "A", np.where(score <= 27, "B", "C")).astype(object)
    ungraded_idx = rng.choice(n, size=int(n * 0.1), replace=False)
    grade[ungraded_idx] = None

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n),
        "score": score,
        "grade": grade,
        "violation_count": violation_count,
        "critical_count": critical_count,
    })
    live = False

print(f"{'Live' if live else 'Offline fallback'} data: {len(inspections_df):,} inspections")
inspections_df.head()

In [ ]:
inspections_df[["score", "violation_count", "critical_count"]].__________()

### Mean, median, and mode

In [ ]:
scores = inspections_df["score"]

manual_mean = scores.sum() / len(scores)
print(f"Manual mean:     {manual_mean:.2f}")
print(f"pandas .mean():  {scores.__________():.2f}")

The **mean** is the sum divided by the count — the familiar "average." It's easy to compute, but it's pulled around by extreme values: a handful of very high scores drag the mean upward even if most inspections are clean.

*Reference figure — for visual intuition, not something you need to memorize.*

![Four different distributions, each with its mean marked by a red line](images/mean-of-distributions.png)

The mean isn't just a formula — it's the balance point of a distribution. All four curves above have very different shapes and centers, but the red line always lands on the same spot: the point where the distribution balances left-to-right.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released under the MIT license.*

In [ ]:
sorted_scores = scores.sort_values().reset_index(drop=True)
mid = len(sorted_scores) // 2
if len(sorted_scores) % 2 == 0:
    manual_median = (sorted_scores[mid - 1] + sorted_scores[mid]) / 2
else:
    manual_median = sorted_scores[mid]

print(f"Manual median:    {manual_median:.2f}")
print(f"pandas .median(): {scores.__________():.2f}")

The **median** is the middle value once the data is sorted — half the inspections score above it, half below. Unlike the mean, a few extremely bad inspections barely move it. If your mean and median are noticeably different (they are here), that's usually a sign of skew — more on that in Part 2.

In [ ]:
manual_mode = scores.value_counts().idxmax()
print(f"Manual mode:    {manual_mode}")
print(f"pandas .mode(): {scores.__________().tolist()}")

# Mode is often more useful for categorical columns than numeric ones:
print(f"\nMost common grade: {inspections_df['grade'].mode().tolist()}")

The **mode** is simply the most frequent value. For a continuous numeric column like `score`, it's rarely the most informative statistic — but for a categorical column like `grade`, it's often exactly what you want ("what's the most common outcome?").

### Variance and standard deviation

In [ ]:
mean_score = scores.mean()
squared_devs = (scores - mean_score) ** 2      # the exact calculation from Lecture 2's lab
manual_variance = squared_devs.sum() / (len(scores) - 1)
manual_std = manual_variance ** 0.5

print(f"Manual variance: {manual_variance:.2f}")
print(f"pandas .var():   {scores.__________():.2f}")
print(f"Manual std dev:  {manual_std:.2f}")
print(f"pandas .std():   {scores.std():.2f}")

`squared_devs` here is exactly the `(score - mean) ** 2` calculation you vectorized by hand in Lecture 2's lab — variance is just the *average* of those squared deviations, and standard deviation is variance's square root, back in the original units. One detail worth being precise about: pandas divides by `n - 1`, not `n` (a correction for estimating from a sample rather than a full population) — that's why the manual formula above uses `len(scores) - 1` too, so the two agree.

*Reference figure — for visual intuition, not something you need to memorize.*

![A bell curve with 1, 2, and 3 standard deviation bands shaded around the mean](images/std-dev-bands.png)

Standard deviation isn't just a number — it's a unit of distance from the mean. For data shaped like this curve, roughly 68% of values fall within 1 standard deviation of the mean (the darkest band), about 95% within 2, and about 99.7% within 3. This is sometimes called the *68-95-99.7 rule*, and it's why a value more than 2-3 standard deviations out already looks unusual before you even apply a formal outlier rule.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released under the MIT license.*

### Quartiles

In [ ]:
q1, q2, q3 = scores.__________([0.25, 0.5, 0.75])
print(f"Q1 (25th percentile):        {q1}")
print(f"Q2 (50th percentile/median): {q2}")
print(f"Q3 (75th percentile):        {q3}")

**Quartiles** split sorted data into four equal-count groups. Q2 is just the median again; Q1 and Q3 mark where the bottom and top quarters begin. The distance between them, `Q3 - Q1`, is the **interquartile range (IQR)** — the width of the "typical" middle 50% of the data, which Part 3 uses to flag outliers.

---

## Part 2 — Distributions

A single number (the mean) can't tell you the *shape* of your data — whether it's symmetric, skewed, or has multiple clusters. For that, you need to look at the whole distribution.

In [ ]:
plt.hist(scores, bins=__________, edgecolor="white")
plt.xlabel("Inspection score")
plt.ylabel("Number of inspections")
plt.title("Distribution of restaurant inspection scores")
plt.show()

A **histogram** groups values into bins and counts how many fall in each. Notice the shape: a longer tail stretching toward higher scores. That's a **right skew** — it's exactly why the mean came out higher than the median in Part 1: a relatively small number of high-violation inspections pull the mean up, but don't move the median much.

### The same plot, with Seaborn

In [ ]:
sns.histplot(data=inspections_df, x="score", kde=__________)
plt.title("Distribution of restaurant inspection scores")
plt.show()

**Seaborn** is a plotting library built directly on top of Pandas DataFrames and Matplotlib — you hand it a `DataFrame` and column names instead of raw arrays, and it comes with better default styling. The `kde=True` overlay adds a smoothed curve tracing the distribution's shape, which makes skew easier to see at a glance than bars alone.

*Reference figure — for visual intuition, not something you need to memorize.*

![A distribution with two separate peaks](images/bimodal-distribution.png)

Not every distribution looks like a single bell curve. This one has two peaks — it's **bimodal**. That shape is often a clue that two different groups got mixed into one column (e.g., "amount of oil in the car" might really be two populations: cars just topped off, and cars way overdue for a change). A histogram is what would reveal this; a mean or median alone would hide it completely.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released under the MIT license.*

---

## Part 3 — Outlier Detection with IQR

A common, principled rule for flagging outliers uses the interquartile range from Part 1: anything more than 1.5×IQR below Q1 or above Q3 is flagged.

In [ ]:
q1, q3 = scores.quantile([0.25, 0.75])
iqr = q3 __________ q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"IQR: {iqr:.1f}")
print(f"Bounds: [{lower_bound:.1f}, {upper_bound:.1f}]")

1.5×IQR is a convention, not a law of nature — it's a widely-used rule of thumb (originating with the box plot) for "unusually far from the middle 50%," not a statistical test proving something is wrong with a data point.

In [ ]:
sns.__________(y=scores)
plt.ylabel("Inspection score")
plt.title("Restaurant inspection scores, with outliers")
plt.show()

A **box plot** draws the box from Q1 to Q3 (the IQR), a line at the median, "whiskers" out to the last point within the 1.5×IQR bounds, and individual dots for everything beyond that — exactly the outliers you just computed.

In [ ]:
outliers = inspections_df[(scores < lower_bound) __________ (scores > upper_bound)]
print(f"{len(outliers):,} inspections flagged as outliers ({len(outliers) / len(inspections_df):.1%})")
outliers[["boro", "cuisine_description", "score", "violation_count"]].sort_values("score", ascending=False).head()

Careful: "outlier" doesn't automatically mean "delete." A restaurant with an unusually high score might be a data-entry mistake — or it might be a genuinely, importantly bad inspection that a health inspector (and a diner) would very much want to know about. IQR flags a point as *worth a second look*; what you do next is a judgment call, not something the formula decides for you.

---

## Part 4 — Correlation vs. Causation

Do any of these numeric columns move together?

In [ ]:
sns.scatterplot(data=inspections_df, x="violation_count", y="score", hue=__________)
plt.title("Score vs. violation count")
plt.show()

Clear upward trend: more violations, higher (worse) score.

In [ ]:
inspections_df[["score", "violation_count", "critical_count"]].__________().round(2)

In [ ]:
sns.heatmap(
    inspections_df[["score", "violation_count", "critical_count"]].corr(),
    annot=__________, cmap="coolwarm", vmin=-1, vmax=1,
)
plt.title("Correlation matrix")
plt.show()

`.corr()` computes the **correlation coefficient** between every pair of numeric columns — a number from -1 (perfectly opposite) to +1 (perfectly aligned), with 0 meaning no linear relationship. A heatmap just makes a whole matrix of these easier to scan at once than a table of numbers.

### Anscombe's Quartet

*Reference figure — for visual intuition, not something you need to memorize.*

![Four scatter plots that all share the same mean, variance, and correlation coefficient, but look completely different](images/anscombe-quartet-panels.png)

These four datasets have *identical* summary statistics: same mean, same variance, same correlation coefficient, even the same best-fit line. Here they are as four separate scatter plots — a clean linear relationship, a curve, a near-perfect line thrown off by one point, and a vertical cluster thrown off by one point.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released under the MIT license.*

![The same four datasets, superimposed on one plot](images/anscombe-quartet-superimposed.png)

Superimposed, the difference is obvious. If you had only looked at `.corr()` or `.describe()` for each of these four datasets, you would have concluded they were basically the same. This is exactly why Parts 1-3 of this lecture never stopped at numbers alone — a chart can show you something a summary statistic hides entirely.

In [ ]:
anscombe_x = {
    "I":   [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    "II":  [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    "III": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    "IV":  [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
}
anscombe_y = {
    "I":   [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68],
    "II":  [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74],
    "III": [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73],
    "IV":  [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
}

for name in anscombe_x:
    x = np.array(anscombe_x[name])
    y = np.array(anscombe_y[name])
    print(f"Dataset {name}: mean(x)={x.mean():.2f}  mean(y)={y.mean():.2f}  "
          f"var(x)={x.var(ddof=1):.2f}  var(y)={y.var(ddof=1):.2f}  "
          f"corr={np.corrcoef(x, y)[0,1]:.3f}")

Not an approximation — these four wildly different-looking datasets really do share the same mean, variance, and correlation, to two or three decimal places. The only way to tell them apart is to actually plot them.

### Correlation is not causation

The classic warning example: ice cream sales and drowning deaths rise and fall together throughout the year, strongly correlated — but ice cream doesn't cause drowning. Both are driven by a third factor: hot weather brings more people to both ice cream stands and swimming pools. That hidden third factor is called a **confounder**.

The `score`/`violation_count` correlation above is a different, subtler case worth naming honestly: it isn't a *discovery* about restaurant hygiene at all. `score` is computed largely **from** violation counts in the first place, so the two being highly correlated is closer to definitional than causal — you'd be surprised if they *weren't* strongly related. Before treating any strong correlation as an interesting finding, it's worth asking: is this relationship telling me something new, or is it just how one of these columns was built from the other?

---

## Part 5 — Choosing the Right Chart

A quick reference for matching a question to a chart type:

| Question | Chart |
|---|---|
| How many rows fall into each category? | Bar chart |
| What's the shape of one numeric column's distribution? | Histogram |
| How does one numeric column compare across categories? | Box plot |
| Is there a relationship between two numeric columns? | Scatter plot |
| How do many numeric columns relate to each other at once? | Heatmap (of `.corr()`) |

The most common mistake isn't picking a "wrong" chart technically — it's picking a chart that *technically* works but obscures the point you're actually trying to make. Part 6 puts this into practice.

---

## Part 6 — Applied EDA and Storytelling, on NYC 311

Same tools, back on the dataset you know best — and this time, aimed at producing a finding you could actually explain to someone who's never opened a notebook. This is exactly what Assignment 2 asks you to do.

### Setup

Same live-pull-with-fallback pattern as before, with `resolution_time_hours` computed the same way as Lecture 4.

In [ ]:
SOCRATA_URL_311 = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

try:
    response = requests.get(
        SOCRATA_URL_311,
        params={
            "$limit": 8000,
            "$order": "created_date DESC",
            "$select": "complaint_type,borough,created_date,closed_date",
        },
        timeout=8,
    )
    response.raise_for_status()
    complaints_df = pd.DataFrame(response.json())
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    # City-wide daily volume dips on weekends -- shared across every borough at once.
    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_weight = np.where(days.dayofweek >= 5, 0.6, 1.0)
    day_weight = day_weight / day_weight.sum()

    day_idx = rng.choice(n_days, size=n, p=day_weight)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")

    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600

print(f"{'Live' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df.head()

In [ ]:
sns.histplot(data=complaints_df, x="__________", kde=True)
plt.title("Distribution of complaint resolution time")
plt.show()

print(complaints_df["resolution_time_hours"].describe())

Same right-skewed shape as the inspection scores in Part 2 — most complaints resolve reasonably quickly, but a long tail drags the mean well above the median. That's the same story worth telling in different data, which is the point: these tools transfer.

In [ ]:
res = complaints_df["resolution_time_hours"].dropna()
q1, q3 = res.quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr

slow_complaints = complaints_df[complaints_df["resolution_time_hours"] __________ upper_bound]
print(f"Bounds: [{lower_bound:.1f}, {upper_bound:.1f}] hours")
print(f"{len(slow_complaints):,} complaints flagged as unusually slow to resolve "
      f"({len(slow_complaints) / len(res):.1%})")

Unlike the restaurant scores, these outliers point somewhere actionable: complaints that took far longer than typical to resolve are exactly the kind of thing a city agency (or a journalist) would want flagged, not quietly dropped.

### Do boroughs' daily complaint volumes move together?

In [ ]:
complaints_df["created_day"] = complaints_df["created_date"].dt.date
daily_by_borough = complaints_df.groupby(["created_day", "borough"]).size().__________(fill_value=0)

sns.heatmap(daily_by_borough.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation of daily complaint counts, across boroughs")
plt.show()

Every borough is positively correlated with every other borough's daily complaint count — a slow day tends to be slow everywhere, a busy day busy everywhere. That's not because a complaint in Brooklyn causes one in Queens; it's a **confounder** again, just like the ice cream example: citywide conditions (day of the week, weather) drive complaint volume across all five boroughs at once.

### Technical storytelling, for Assignment 2

Assignment 2 asks for an EDA and visualization report written for a non-technical audience. A simple framework:

1. **Lead with the finding** — the sentence a busy reader would actually remember, not the method.
2. **One supporting chart** — the smallest chart that proves the finding, not every chart you made along the way.
3. **No jargon** — "IQR," "correlation coefficient," and "groupby" mean nothing to this reader. Say what you found in plain language instead.
4. **State the "so what"** — why should this reader care? What would they do differently knowing this?

In [ ]:
avg_resolution_by_borough = (
    complaints_df.groupby("borough")["resolution_time_hours"].mean().sort_values(ascending=__________)
)

avg_resolution_by_borough.plot(kind="barh")
plt.xlabel("Average hours to resolve a complaint")
plt.title("How long does it take to resolve a 311 complaint, by borough?")
plt.gca().invert_yaxis()
plt.show()

print(avg_resolution_by_borough.round(1))

**Applying the framework:**

*(Fill in the bracketed borough names using your own output from the chart above — the live-pull data changes run to run, so the answer isn't fixed.)*

> On average, complaints in [slowest borough] take the longest to resolve — noticeably longer than in [fastest borough]. If you've filed a 311 complaint and it feels like it's taking forever, where you live may genuinely be part of why.

Notice what that summary leaves out: no mention of `groupby`, no p-values, no mention of the underlying dataset size. One chart, one finding, one reason to care.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook: `lect05_eda_statistics_exercise.ipynb`.

---

## Part 7 — Cheat Sheet

| Task | Code |
|---|---|
| Mean / median / mode | `.mean()` / `.median()` / `.mode()` |
| Variance / standard deviation | `.var()` / `.std()` |
| Quartiles | `.quantile([0.25, 0.5, 0.75])` |
| IQR outlier bounds | `q1 - 1.5*iqr`, `q3 + 1.5*iqr` |
| Histogram | `sns.histplot(data=df, x="col", kde=True)` |
| Box plot | `sns.boxplot(y=df["col"])` |
| Scatter plot | `sns.scatterplot(data=df, x="a", y="b", hue="c")` |
| Correlation matrix | `df[cols].corr()` |
| Heatmap | `sns.heatmap(df[cols].corr(), annot=True)` |

---

## Part 8 — Key Terms

- **Mean**: the sum of values divided by the count; sensitive to extreme values.
- **Median**: the middle value once sorted; resistant to extreme values.
- **Mode**: the most frequently occurring value.
- **Variance / standard deviation**: measures of how spread out values are around the mean.
- **Quartile / IQR**: the values splitting data into four equal-count groups; `Q3 - Q1` is the interquartile range.
- **Skew**: an asymmetric distribution, with a longer tail on one side (often visible as mean ≠ median).
- **Outlier**: a value unusually far from the rest of the data, by some rule (e.g., 1.5×IQR beyond Q1/Q3).
- **Correlation coefficient**: a number from -1 to +1 describing how strongly two numeric variables move together linearly.
- **Confounder**: a hidden third factor that drives two variables to correlate without either causing the other.